In [ ]:
import pandas as pd
import time
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from webdriver_manager.chrome import ChromeDriverManager
from selenium.webdriver.chrome.options import Options

# Load URLs
df = pd.read_csv("Sample_input_links.csv")
urls = df["url"].dropna().unique()

# Chrome options
options = Options()
options.add_argument("--start-maximized")
options.add_argument("--disable-blink-features=AutomationControlled")

driver = webdriver.Chrome(service=Service(ChromeDriverManager().install()), options=options)

final_results = []

for url in urls:
    print("Scraping:", url)
    driver.get(url)

    wait = WebDriverWait(driver, 20)

    booth_list = []

    # TRY 3 times in case content loads late
    for attempt in range(3):
        try:
            # Scroll to trigger lazy loading
            driver.execute_script("window.scrollBy(0, document.body.scrollHeight);")
            time.sleep(3)

            elements = wait.until(
                EC.presence_of_all_elements_located(
                    (By.CSS_SELECTOR, "a.imc-exhibitorcard--link.imc-link--hover-underline")
                )
            )

            booth_list = [e.text.strip() for e in elements if e.text.strip()]

            if booth_list:
                break

        except:
            print(f"Retry {attempt+1} for:", url)

    # Fallback selector (in rare UI changes)
    if not booth_list:
        fallback = driver.find_elements(By.XPATH, "//a[contains(@class,'imc-exhibitorcard--link')]")
        booth_list = [f.text.strip() for f in fallback if f.text.strip()]

    # Final safeguard
    if booth_list:
        joined = " ||| ".join(booth_list)
    else:
        joined = "NOT FOUND"

    final_results.append([url, joined])

driver.quit()

# Save
pd.DataFrame(final_results, columns=["url", "booth_numbers"]).to_csv("final_output.csv", index=False)
print("✅ EXTRACTION COMPLETE")

Scraping: https://www.atlantamarket.com/exhibitor/103027/line/4a2a2eb4-3c45-f59f-c000-2218d474c74a
Scraping: https://www.atlantamarket.com/exhibitor/123834/line/1c1f42e1-170e-b077-9b65-6bf9cd0fbd5b
Scraping: https://www.atlantamarket.com/exhibitor/126304/line/a950b38e-06f6-9690-88c1-dbdb9358abe2
Scraping: https://www.atlantamarket.com/exhibitor/120011/line/68c757c8-9559-41af-7d00-cb20d593026c
Scraping: https://www.atlantamarket.com/exhibitor/104130/line/6d5f31db-59c5-48bc-87f6-35bddc10d3cf
Scraping: https://www.atlantamarket.com/exhibitor/126156/line/fc5944c3-33d8-cd57-5160-8c15ed0f2f32
Scraping: https://www.atlantamarket.com/exhibitor/100264/line/084faa23-7a71-484b-c412-6af144f03c61
Scraping: https://www.atlantamarket.com/exhibitor/123834/line/6ce8f427-46b3-dc8d-6d66-432f3e7faab3
Scraping: https://www.atlantamarket.com/exhibitor/100184/line/457d1420-27c6-d444-81d2-b07b99c8bc3b
Scraping: https://www.atlantamarket.com/exhibitor/126304/line/b24995ec-a969-732c-7721-929edcd74f13
Scraping: 